In [4]:
import os
import glob
import mne
import numpy as np
import pandas as pd
from mne.datasets import fetch_hcp_mmp_parcellation
from mne_connectivity import envelope_correlation
from mne.minimum_norm import read_inverse_operator  
from mne.minimum_norm import apply_inverse_raw
import mne_connectivity

from mne_connectivity import spectral_connectivity_epochs

from concurrent.futures import ProcessPoolExecutor


In [5]:
# --- Base and Subdirectories ---
base_dir = "/scratch2/alinat/project/PD-EEG-ANON_eegOnly/derivatives"
mne_pipeline_dir = f"{base_dir}/mne_pipeline_eye-closed_run1"
fs_subject_dir = f"{base_dir}/FS_SUBJECT_DIR/fsaverage1/bem"

# --- Dynamically Find Subjects and Sessions ---
subject_session_paths = sorted(glob.glob(os.path.join(mne_pipeline_dir, "sub-*/ses-*/eeg")))
# subject_sessions = {
#     os.path.basename(os.path.dirname(os.path.dirname(path))):  # Extract subject name
#     os.path.basename(os.path.dirname(path))  # Extract session name
#     for path in subject_session_paths
# }

subjects = sorted(set([i.split('/')[-3] for i in subject_session_paths]))
sessions = sorted(set([i.split('/')[-2] for i in subject_session_paths]))

In [ ]:
# print(len(subjects))
# print(len(sessions))
# print(len(subject_session_paths))

In [6]:
# --- Frequency Bands ---
bands = {
    "delta": [0.5, 4],
    "theta": [4.01, 7.99],
    "alpha": [8, 12.99],
    "beta": [13, 30],
    "gamma1": [30.01, 49],
    'gamma2': [51, 80],
}

# Fetch the full HCP-MMP atlas (360 labels)
fetch_hcp_mmp_parcellation(subjects_dir=(('/').join(fs_subject_dir.split('/')[:-2])), combine=False, accept=True)

# Load the labels from the annotation file
hcp_labels = mne.read_labels_from_annot(
    subject="fsaverage1",
    parc="HCPMMP1",
    subjects_dir=(('/').join(fs_subject_dir.split('/')[:-2])),
)

# Extract region names for headers
headers = [label.name for label in hcp_labels]



Reading labels from parcellation...
   read 181 labels from /scratch2/alinat/project/PD-EEG-ANON_eegOnly/derivatives/FS_SUBJECT_DIR/fsaverage1/label/lh.HCPMMP1.annot
   read 181 labels from /scratch2/alinat/project/PD-EEG-ANON_eegOnly/derivatives/FS_SUBJECT_DIR/fsaverage1/label/rh.HCPMMP1.annot


In [ ]:
#AEC

# Define a function to compute functional connectivity for a specific band
def compute_connectivity_for_band(band_info, epochs, inverse_operators, hcp_labels, session_dir, headers, subject, session):
    band, freqs = band_info
    band_epochs = epochs.copy().filter(l_freq=freqs[0], h_freq=freqs[1], n_jobs=1)

    for name, inv_op in inverse_operators.items():
        # Apply inverse operator to band-pass-filtered epochs to get SourceEstimate objects
        stcs = mne.minimum_norm.apply_inverse_epochs(band_epochs, inv_op, lambda2=1.0 / 9.0, pick_ori="normal")

        # Extract label time courses
        label_ts = mne.extract_label_time_course(
            stcs,
            hcp_labels,
            inv_op["src"],
            mode="mean_flip",
            return_generator=False,
        )

        # Pairwise Orthogonalization
        corr_obj_pairwise = envelope_correlation(
            label_ts,
            orthogonalize="pairwise"
        )
        corr_matrix_pairwise = corr_obj_pairwise.combine().get_data(output="dense")[:, :, 0]

        # Save pairwise correlation matrix with headers
        pairwise_filename = os.path.join(
            session_dir,
            f"{subject}_{session}_{name}_aec_{band}.csv"
        )
        df_pairwise = pd.DataFrame(data=corr_matrix_pairwise, index=headers, columns=headers)
        df_pairwise.to_csv(pairwise_filename)

# --- Process Each Subject and Session ---
for subject in subjects:
    for session in sessions:
        session_dir = os.path.join(mne_pipeline_dir, subject, session, "eeg")

        if os.path.exists(session_dir):
            epochs_path = os.path.join(
                session_dir, f"{subject}_{session}_task-restEyesClosed_proc-clean_badCh-interp_badEp-marked_epo.fif"
            )
            fwd_path = f"{fs_subject_dir}/{subject}_{session}-fwd.fif"
            ad_hoc_inv_path = f"{fs_subject_dir}/{subject}_{session}_ad-hoc_inv.fif"
            diagonal_inv_path = f"{fs_subject_dir}/{subject}_{session}_diagonal_inv.fif"

            # Load Data
            epochs = mne.read_epochs(epochs_path, preload=True)
            fwd = mne.read_forward_solution(fwd_path)
            ad_hoc_inv = read_inverse_operator(ad_hoc_inv_path)
            diagonal_inv = read_inverse_operator(diagonal_inv_path)

            # Apply average reference
            epochs, _ = mne.set_eeg_reference(epochs, ref_channels="average", projection=True)

            # Define Inverse Operators
            inverse_operators = {"ad-hoc": ad_hoc_inv, "diagonal": diagonal_inv}

            # Prepare input for parallel processing
            band_items = bands.items()

            # Run computations in parallel for each band
            with ProcessPoolExecutor() as executor:
                executor.map(
                    compute_connectivity_for_band,
                    band_items,
                    [epochs] * len(bands),
                    [inverse_operators] * len(bands),
                    [hcp_labels] * len(bands),
                    [session_dir] * len(bands),
                    [headers] * len(bands),
                    [subject] * len(bands),
                    [session] * len(bands)
                )


Reading /scratch2/alinat/project/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline_eye-closed_run1/sub-NZ000052/ses-20180209/eeg/sub-NZ000052_ses-20180209_task-restEyesClosed_proc-clean_badCh-interp_badEp-marked_epo.fif ...
    Read a total of 1 projection items:
        Average EEG reference (1 x 54) active
    Found the data of interest:
        t =       0.00 ...    3000.00 ms
        0 CTF compensation matrices available
Not setting metadata
397 matching events found
No baseline correction applied
Created an SSP operator (subspace dimension = 1)
1 projection items activated
Reading forward solution from /scratch2/alinat/project/PD-EEG-ANON_eegOnly/derivatives/FS_SUBJECT_DIR/fsaverage1/bem/sub-NZ000052_ses-20180209-fwd.fif...
    Reading a source space...
    [done]
    Reading a source space...
    [done]
    2 source spaces read
    Desired named matrix (kind = 3523 (FIFF_MNE_FORWARD_SOLUTION_GRAD)) not available
    Read EEG forward solution (8196 sources, 58 channels, free orientatio

/tmp/alinat/ipykernel_3034556/464649826.py:4: RuntimeWarning: filter_length (1651) is longer than the signal (751), distortion is likely. Reduce filter length or filter a longer signal.
  band_epochs = epochs.copy().filter(l_freq=freqs[0], h_freq=freqs[1], n_jobs=1)


Setting up band-pass filter from 4 - 8 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 4.01
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 3.01 Hz)
- Upper passband edge: 7.99 Hz
- Upper transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 8.99 Hz)
- Filter length: 413 samples (1.652 s)

Setting up band-pass filter from 8 - 13 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 12.99 Hz
- Upper transition bandwidth: 3.25 Hz (-6 dB cutoff frequency: 14.61 Hz)
- Filter 

/tmp/alinat/ipykernel_3034556/464649826.py:4: RuntimeWarning: filter_length (1651) is longer than the signal (751), distortion is likely. Reduce filter length or filter a longer signal.
  band_epochs = epochs.copy().filter(l_freq=freqs[0], h_freq=freqs[1], n_jobs=1)


Setting up band-pass filter from 4 - 8 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 4.01
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 3.01 Hz)
- Upper passband edge: 7.99 Hz
- Upper transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 8.99 Hz)
- Filter length: 413 samples (1.652 s)

Setting up band-pass filter from 8 - 13 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 12.99 Hz
- Upper transition bandwidth: 3.25 Hz (-6 dB cutoff frequency: 14.61 Hz)
- Filter 

In [ ]:
#dwPLI

# Define a function to compute wPLI2_debiased connectivity for a specific band
def compute_wpli2_for_band(band_info, epochs, inverse_operators, hcp_labels, session_dir, headers, subject, session):
    band, freqs = band_info
    band_epochs = epochs.copy().filter(l_freq=freqs[0], h_freq=freqs[1], n_jobs=1)

    for name, inv_op in inverse_operators.items():
        # Apply inverse operator to band-pass-filtered epochs to get SourceEstimate objects
        stcs = mne.minimum_norm.apply_inverse_epochs(band_epochs, inv_op, lambda2=1.0 / 9.0, pick_ori="normal")

        # Extract label time courses
        label_ts = mne.extract_label_time_course(
            stcs,
            hcp_labels,
            inv_op["src"],
            mode="mean_flip",
            return_generator=False,
        )

        # --- wPLI2_debiased Calculation ---
        conn_wpli2_debiased_epochs = spectral_connectivity_epochs(
            data=label_ts,
            method="wpli2_debiased",  # Weighted Phase Lag Index (debiased)
            sfreq=band_epochs.info["sfreq"],
            fmin=freqs[0],
            fmax=freqs[1],
            mode="fourier",
            faverage=True,
            n_jobs=1,  # Use parallel processing for spectral connectivity
        ).get_data(output="dense")[:, :, 0]

        # Save wPLI2_debiased matrix
        wpli2_debiased_filename = os.path.join(
            session_dir,
            f"{subject}_{session}_{name}_dwpli2_{band}.csv",
        )
        df_wpli2_debiased = pd.DataFrame(
            data=conn_wpli2_debiased_epochs, index=headers, columns=headers
        )
        df_wpli2_debiased.to_csv(wpli2_debiased_filename)

# --- Process Each Subject and Session ---
for subject in subjects:
    for session in sessions:
        session_dir = os.path.join(mne_pipeline_dir, subject, session, "eeg")

        if os.path.exists(session_dir):
            epochs_path = os.path.join(
                session_dir, f"{subject}_{session}_task-restEyesClosed_proc-clean_badCh-interp_badEp-marked_epo.fif"
            )
            fwd_path = f"{fs_subject_dir}/{subject}_{session}-fwd.fif"
            ad_hoc_inv_path = f"{fs_subject_dir}/{subject}_{session}_ad-hoc_inv.fif"
            diagonal_inv_path = f"{fs_subject_dir}/{subject}_{session}_diagonal_inv.fif"

            # Load Data
            epochs = mne.read_epochs(epochs_path, preload=True)
            fwd = mne.read_forward_solution(fwd_path)
            ad_hoc_inv = read_inverse_operator(ad_hoc_inv_path)
            diagonal_inv = read_inverse_operator(diagonal_inv_path)

            # Apply average reference
            epochs, _ = mne.set_eeg_reference(epochs, ref_channels="average", projection=True)

            # Define Inverse Operators
            inverse_operators = {"ad-hoc": ad_hoc_inv, "diagonal": diagonal_inv}

            # Prepare input for parallel processing
            band_items = bands.items()

            # Run computations in parallel for each band
            with ProcessPoolExecutor() as executor:
                executor.map(
                    compute_wpli2_for_band,
                    band_items,
                    [epochs] * len(bands),
                    [inverse_operators] * len(bands),
                    [hcp_labels] * len(bands),
                    [session_dir] * len(bands),
                    [headers] * len(bands),
                    [subject] * len(bands),
                    [session] * len(bands)
                )


In [ ]:

# # --- Process Each Subject and Session ---
# for subject in subjects:
#     for session in sessions:

#         session_dir = os.path.join(mne_pipeline_dir, subject, session, "eeg")

#         if os.path.exists(session_dir):
#             #
#             #raw_path = os.path.join(session_dir, f"{subject}_{session}_task-restEyesClosed_run-01_proc-clean_badCh-interp_raw.fif",)
#             epochs_path = os.path.join(session_dir, f"{subject}_{session}_task-restEyesClosed_proc-clean_badCh-interp_badEp-marked_epo.fif",)
#             fwd_path = f"{fs_subject_dir}/{subject}_{session}-fwd.fif"
#             ad_hoc_inv_path = f"{fs_subject_dir}/{subject}_{session}_ad-hoc_inv.fif"
#             diagonal_inv_path = f"{fs_subject_dir}/{subject}_{session}_diagonal_inv.fif"

#             # # --- Check for Missing Data ---
#             # if not os.path.exists(raw_path) or not os.path.exists(epochs_path):
#             #     print(f"Skipping {subject}, {session} as data files are missing.")
#             #     continue

#             # --- Load Data ---
#             #raw = mne.io.read_raw_fif(raw_path, preload=True)
#             epochs = mne.read_epochs(epochs_path, preload=True)
#             fwd = mne.read_forward_solution(fwd_path)
#             ad_hoc_inv = read_inverse_operator(ad_hoc_inv_path)
#             diagonal_inv = read_inverse_operator(diagonal_inv_path)

#             ## Apply average reference and reset the custom reference flag
#             epochs, _ = mne.set_eeg_reference(epochs, ref_channels="average", projection=True)

#             # Define Inverse Operators
#             inverse_operators = {"ad-hoc": ad_hoc_inv, "diagonal": diagonal_inv}

#             # # --- Create and Save Inverse Solutions for Raw ---
#             # for name, inv_op in inverse_operators.items():
#             #     stc = apply_inverse_raw(
#             #         raw, inv_op, lambda2=1.0 / 9.0, method="dSPM", pick_ori="normal"
#             #     )
#             #     stc.save(os.path.join(session_dir, f"{subject}_{session}_raw_{name}_stc"), overwrite=True)

#             # # --- Create and Save Inverse Solutions for Epochs ---
#             # for name, inv_op in inverse_operators.items():
#             #     # Create folder for epoch inverse solutions
#             #     inv_sol_epochs_dir = os.path.join(session_dir, "inv_sol_epoch")
#             #     os.makedirs(inv_sol_epochs_dir, exist_ok=True)

#             #     # Compute inverse solutions for epochs
#             #     stcs = mne.minimum_norm.apply_inverse_epochs(
#             #         epochs, inv_op, lambda2=1.0 / 9.0, pick_ori="normal"
#             #     )
#             #     for i, stc in enumerate(stcs):
#             #         stc.save(
#             #             os.path.join(
#             #                 inv_sol_epochs_dir, f"{subject}_{session}_epo_{name}_epoch-{i + 1}_stc"
#             #             ),
#             #             overwrite=True,
#             #         )

#             # --- Apply Bandpass Filtering and Compute Functional Connectivity ---
#             for band, freqs in bands.items():

#                 band_epochs = epochs.copy().filter(l_freq=freqs[0], h_freq=freqs[1], n_jobs=-1)

#                 for name, inv_op in inverse_operators.items():

#                     # Apply inverse operator to band-pass-filtered epochs to get SourceEstimate objects
#                     stcs = mne.minimum_norm.apply_inverse_epochs(band_epochs, inv_op, lambda2=1.0 / 9.0, pick_ori="normal")

#                     # Extract label time courses
#                     label_ts = mne.extract_label_time_course(stcs, 
#                                                              hcp_labels, 
#                                                              inv_op["src"], 
#                                                              mode="mean_flip", 
#                                                              return_generator=False,
#                                                              #allow_empty = True,
#                                                              )

#                     # Pairwise Orthogonalization
#                     corr_obj_pairwise = envelope_correlation(label_ts,  # Use the extracted time series directly
#                                                              orthogonalize="pairwise"
#                                                              )
#                     corr_matrix_pairwise = corr_obj_pairwise.combine().get_data(output="dense")[:, :, 0]

#                     # Save pairwise correlation matrix with headers
#                     pairwise_filename = os.path.join(session_dir,
#                                                      f"{subject}_{session}_{name}_pairwise-con_{band}.csv"
#                                                      )
#                     df_pairwise = pd.DataFrame(data=corr_matrix_pairwise, index=headers, columns=headers)
#                     df_pairwise.to_csv(pairwise_filename)


#                     # # Symmetric Orthogonalization
#                     # label_ts_orth = mne_connectivity.envelope.symmetric_orth(label_ts)
#                     # corr_obj_symmetric = envelope_correlation(
#                     #     (mne.filter.filter_data(ts, band_epochs.info["sfreq"], freqs[0], freqs[1])
#                     #     for ts in label_ts_orth),
#                     #     orthogonalize=False  # Symmetric orthogonalization already applied
#                     # )
#                     # corr_matrix_symmetric = corr_obj_symmetric.combine().get_data(output="dense")[:, :, 0]

#                     # # Save symmetric correlation matrix with headers
#                     # symmetric_filename = os.path.join(
#                     #     session_dir,
#                     #     f"{subject}_{session}_{name}_symmetric-con_{band}.csv"
#                     # )
#                     # df_symmetric = pd.DataFrame(data=corr_matrix_symmetric, index=headers, columns=headers)
#                     # df_symmetric.to_csv(symmetric_filename)


#                     # # --- wPLI Calculation ---
#                     # conn_wpli_epochs = spectral_connectivity_epochs(
#                     #     data=label_ts,
#                     #     method="wpli",  # Weighted Phase Lag Index
#                     #     sfreq=band_epochs.info["sfreq"],
#                     #     fmin=freqs[0],
#                     #     fmax=freqs[1],
#                     #     mode="multitaper",  # Can use multitaper or fourier
#                     #     faverage=True,  # Average within the frequency band
#                     # ).get_data(output="dense")[:, :, 0]

#                     # # Save wPLI matrix
#                     # wpli_filename = os.path.join(
#                     #     session_dir,
#                     #     f"{subject}_{session}_{name}_wpli_{band}.csv"
#                     # )
#                     # df_wpli = pd.DataFrame(data=conn_wpli_epochs, index=headers, columns=headers)
#                     # df_wpli.to_csv(wpli_filename)

#                     # --- wPLI2_debiased Calculation ---
#                     # conn_wpli2_debiased_epochs = spectral_connectivity_epochs(
#                     #     data=label_ts,
#                     #     method="wpli2_debiased",  # Weighted Phase Lag Index (debiased)
#                     #     sfreq=band_epochs.info["sfreq"],
#                     #     fmin=freqs[0],
#                     #     fmax=freqs[1],
#                     #     mode="multitaper",
#                     #     faverage=True,
#                     #     n_jobs=100,
#                     # ).get_data(output="dense")[:, :, 0]

#                     # # Save wPLI2_debiased matrix
#                     # wpli2_debiased_filename = os.path.join(session_dir,
#                     #                                        f"{subject}_{session}_{name}_wpli2-debiased_{band}.csv")
#                     # df_wpli2_debiased = pd.DataFrame(data=conn_wpli2_debiased_epochs, index=headers, columns=headers)
#                     # df_wpli2_debiased.to_csv(wpli2_debiased_filename)            



In [ ]:

# # --- Process Each Subject and Session ---
# for subject in subjects:
#     for session in sessions:

#         session_dir = os.path.join(mne_pipeline_dir, subject, session, "eeg")

#         if os.path.exists(session_dir):
#             #
#             #raw_path = os.path.join(session_dir, f"{subject}_{session}_task-restEyesClosed_run-01_proc-clean_badCh-interp_raw.fif",)
#             epochs_path = os.path.join(session_dir, f"{subject}_{session}_task-restEyesClosed_proc-clean_badCh-interp_badEp-marked_epo.fif",)
#             fwd_path = f"{fs_subject_dir}/{subject}_{session}-fwd.fif"
#             ad_hoc_inv_path = f"{fs_subject_dir}/{subject}_{session}_ad-hoc_inv.fif"
#             diagonal_inv_path = f"{fs_subject_dir}/{subject}_{session}_diagonal_inv.fif"

#             # # --- Check for Missing Data ---
#             # if not os.path.exists(raw_path) or not os.path.exists(epochs_path):
#             #     print(f"Skipping {subject}, {session} as data files are missing.")
#             #     continue

#             # --- Load Data ---
#             #raw = mne.io.read_raw_fif(raw_path, preload=True)
#             epochs = mne.read_epochs(epochs_path, preload=True)
#             fwd = mne.read_forward_solution(fwd_path)
#             ad_hoc_inv = read_inverse_operator(ad_hoc_inv_path)
#             diagonal_inv = read_inverse_operator(diagonal_inv_path)

#             ## Apply average reference and reset the custom reference flag
#             epochs, _ = mne.set_eeg_reference(epochs, ref_channels="average", projection=True)

#             # Define Inverse Operators
#             inverse_operators = {"ad-hoc": ad_hoc_inv, "diagonal": diagonal_inv}

#             # # --- Create and Save Inverse Solutions for Raw ---
#             # for name, inv_op in inverse_operators.items():
#             #     stc = apply_inverse_raw(
#             #         raw, inv_op, lambda2=1.0 / 9.0, method="dSPM", pick_ori="normal"
#             #     )
#             #     stc.save(os.path.join(session_dir, f"{subject}_{session}_raw_{name}_stc"), overwrite=True)

#             # # --- Create and Save Inverse Solutions for Epochs ---
#             # for name, inv_op in inverse_operators.items():
#             #     # Create folder for epoch inverse solutions
#             #     inv_sol_epochs_dir = os.path.join(session_dir, "inv_sol_epoch")
#             #     os.makedirs(inv_sol_epochs_dir, exist_ok=True)

#             #     # Compute inverse solutions for epochs
#             #     stcs = mne.minimum_norm.apply_inverse_epochs(
#             #         epochs, inv_op, lambda2=1.0 / 9.0, pick_ori="normal"
#             #     )
#             #     for i, stc in enumerate(stcs):
#             #         stc.save(
#             #             os.path.join(
#             #                 inv_sol_epochs_dir, f"{subject}_{session}_epo_{name}_epoch-{i + 1}_stc"
#             #             ),
#             #             overwrite=True,
#             #         )

#             # --- Apply Bandpass Filtering and Compute Functional Connectivity ---
#             for band, freqs in bands.items():

#                 band_epochs = epochs.copy().filter(l_freq=freqs[0], h_freq=freqs[1], n_jobs=5)

#                 for name, inv_op in inverse_operators.items():

#                     # Apply inverse operator to band-pass-filtered epochs to get SourceEstimate objects
#                     stcs = mne.minimum_norm.apply_inverse_epochs(band_epochs, inv_op, lambda2=1.0 / 9.0, pick_ori="normal")

#                     # Extract label time courses
#                     label_ts = mne.extract_label_time_course(stcs, 
#                                                              hcp_labels, 
#                                                              inv_op["src"], 
#                                                              mode="mean_flip", 
#                                                              return_generator=False
#                                                              )

#                     # Pairwise Orthogonalization
#                     # corr_obj_pairwise = envelope_correlation(label_ts,  # Use the extracted time series directly
#                     #                                          orthogonalize="pairwise"
#                     #                                          )
#                     # corr_matrix_pairwise = corr_obj_pairwise.combine().get_data(output="dense")[:, :, 0]

#                     # # Save pairwise correlation matrix with headers
#                     # pairwise_filename = os.path.join(session_dir,
#                     #                                  f"{subject}_{session}_{name}_pairwise-con_{band}.csv"
#                     #                                  )
#                     # df_pairwise = pd.DataFrame(data=corr_matrix_pairwise, index=headers, columns=headers)
#                     # df_pairwise.to_csv(pairwise_filename)


#                     # # Symmetric Orthogonalization
#                     # label_ts_orth = mne_connectivity.envelope.symmetric_orth(label_ts)
#                     # corr_obj_symmetric = envelope_correlation(
#                     #     (mne.filter.filter_data(ts, band_epochs.info["sfreq"], freqs[0], freqs[1])
#                     #     for ts in label_ts_orth),
#                     #     orthogonalize=False  # Symmetric orthogonalization already applied
#                     # )
#                     # corr_matrix_symmetric = corr_obj_symmetric.combine().get_data(output="dense")[:, :, 0]

#                     # # Save symmetric correlation matrix with headers
#                     # symmetric_filename = os.path.join(
#                     #     session_dir,
#                     #     f"{subject}_{session}_{name}_symmetric-con_{band}.csv"
#                     # )
#                     # df_symmetric = pd.DataFrame(data=corr_matrix_symmetric, index=headers, columns=headers)
#                     # df_symmetric.to_csv(symmetric_filename)


#                     # # --- wPLI Calculation ---
#                     # conn_wpli_epochs = spectral_connectivity_epochs(
#                     #     data=label_ts,
#                     #     method="wpli",  # Weighted Phase Lag Index
#                     #     sfreq=band_epochs.info["sfreq"],
#                     #     fmin=freqs[0],
#                     #     fmax=freqs[1],
#                     #     mode="multitaper",  # Can use multitaper or fourier
#                     #     faverage=True,  # Average within the frequency band
#                     # ).get_data(output="dense")[:, :, 0]

#                     # # Save wPLI matrix
#                     # wpli_filename = os.path.join(
#                     #     session_dir,
#                     #     f"{subject}_{session}_{name}_wpli_{band}.csv"
#                     # )
#                     # df_wpli = pd.DataFrame(data=conn_wpli_epochs, index=headers, columns=headers)
#                     # df_wpli.to_csv(wpli_filename)

#                     # --- wPLI2_debiased Calculation ---
#                     conn_wpli2_debiased_epochs = spectral_connectivity_epochs(
#                         data=label_ts,
#                         method="wpli2_debiased",  # Weighted Phase Lag Index (debiased)
#                         sfreq=band_epochs.info["sfreq"],
#                         fmin=freqs[0],
#                         fmax=freqs[1],
#                         mode="fourier",#"multitaper",
#                         faverage=True,
#                         n_jobs=100,
#                     ).get_data(output="dense")[:, :, 0]

#                     # Save wPLI2_debiased matrix
#                     wpli2_debiased_filename = os.path.join(session_dir,
#                                                            f"{subject}_{session}_{name}_wpli2-debiased_{band}.csv")
#                     df_wpli2_debiased = pd.DataFrame(data=conn_wpli2_debiased_epochs, index=headers, columns=headers)
#                     df_wpli2_debiased.to_csv(wpli2_debiased_filename)            

